# DistilBERT (Colab) — Experiment E (Loss Strategy Sweep)

This notebook runs a 5-run sweep with fixed preprocessing/hyperparameters and varying `loss_strategy`:

- `bce_no_pos_weight`
- `bce_pos_weight_clip20`
- `focal_gamma1`
- `focal_gamma2`
- `focal_gamma3`

Core setup is fixed per run:

- `MAX_LENGTH=128`, `BATCH_SIZE=32`, `GRADIENT_ACCUMULATION_STEPS=1`, `MAX_EPOCHS=5`
- `LR=1.75e-5`, `WEIGHT_DECAY=0.015`, `WARMUP_RATIO=0.1`, `MAX_GRAD_NORM=1.0`
- early stop: `patience=2`, `min_delta=0.0`
- `USE_BF16=True`, `USE_TORCH_COMPILE=False`, `NUM_WORKERS=2`
- split: `validation_fraction=0.1`, `random_state=42`, `use_iterative_stratify=True`
- preprocessing: `rebalance_train=False`
- threshold tuning grid: `np.arange(0.05, 0.951, 0.01)`

In [ ]:
# Colab setup: install dependencies
!pip -q install torch transformers pandas numpy matplotlib scikit-learn seaborn iterative-stratification

In [ ]:
# Mount Google Drive and set paths
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'artifacts_colab_exp_e'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import contextlib
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix, f1_score

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels, threshold_grid: np.ndarray):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 3), 'best_f1_on_val': best_f1})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 20.0) -> torch.Tensor:
    y = np.asarray(y_train, dtype=np.float32)
    positives = y.sum(axis=0)
    total = float(y.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    gamma: float,
    pos_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none', pos_weight=pos_weight)
    probs = torch.sigmoid(logits)
    pt = probs * targets + (1.0 - probs) * (1.0 - targets)
    focal_factor = (1.0 - pt).pow(gamma)
    return (focal_factor * bce).mean()


def make_confusion_artifacts(y_true: np.ndarray, y_pred: np.ndarray, labels):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    per_label_rows = []
    for j, label in enumerate(labels):
        cm = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        per_label_rows.append({'label': label, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})

    per_label_df = pd.DataFrame(per_label_rows)

    agg_cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=[0, 1])
    agg_tn, agg_fp, agg_fn, agg_tp = agg_cm.ravel()
    aggregate_df = pd.DataFrame([
        {'tn': int(agg_tn), 'fp': int(agg_fp), 'fn': int(agg_fn), 'tp': int(agg_tp)}
    ])

    return per_label_df, aggregate_df, agg_cm


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


def enc_dict(enc):
    keys = [k for k in ('input_ids', 'attention_mask') if k in enc]
    return {k: enc[k] for k in keys}

In [ ]:
# Experiment E configuration
DEVICE = pick_device()
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.manual_seed(42)
np.random.seed(42)

MODEL_NAME = 'distilbert-base-uncased'
PROBLEM_TYPE = 'multi_label_classification'

MAX_LENGTH = 128
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
MAX_EPOCHS = 5

LR = 1.75e-5
WEIGHT_DECAY = 0.015
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 0.0

USE_BF16 = True
USE_TORCH_COMPILE = False
NUM_WORKERS = 2

VALIDATION_FRACTION = 0.1
RANDOM_STATE = 42
USE_ITERATIVE_STRATIFY = True

REBALANCE_TRAIN = False
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
SKIP_COMPLETED = True

LOSS_VARIANTS = [
    {'loss_strategy': 'bce_no_pos_weight', 'loss_type': 'bce', 'use_pos_weight': False, 'gamma': None},
    {'loss_strategy': 'bce_pos_weight_clip20', 'loss_type': 'bce', 'use_pos_weight': True, 'gamma': None},
    {'loss_strategy': 'focal_gamma1', 'loss_type': 'focal', 'use_pos_weight': False, 'gamma': 1.0},
    {'loss_strategy': 'focal_gamma2', 'loss_type': 'focal', 'use_pos_weight': False, 'gamma': 2.0},
    {'loss_strategy': 'focal_gamma3', 'loss_type': 'focal', 'use_pos_weight': False, 'gamma': 3.0},
]

_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('USE_BF16 requested but bf16 not supported on this GPU; training in full precision.')

print('Device:', DEVICE)
print('AMP (bf16):', USE_AMP)
print('TF32 enabled on CUDA:', DEVICE.type == 'cuda')
print('Total runs:', len(LOSS_VARIANTS))

In [ ]:
# Run Experiment E sweep: fixed preprocessing + 5 loss strategies
summary_rows = []
per_label_frames = []
threshold_frames = []
conf_per_label_base_frames = []
conf_per_label_tuned_frames = []
conf_agg_base_rows = []
conf_agg_tuned_rows = []

pin = DEVICE.type == 'cuda'

for idx, spec in enumerate(LOSS_VARIANTS, start=1):
    run_id = spec['loss_strategy']
    row_file = ARTIFACT_DIR / f'summary_{run_id}.csv'
    per_label_file = ARTIFACT_DIR / f'per_label_{run_id}.csv'
    thresholds_file = ARTIFACT_DIR / f'thresholds_{run_id}.csv'
    conf_per_label_base_file = ARTIFACT_DIR / f'confusion_per_label_baseline_{run_id}.csv'
    conf_per_label_tuned_file = ARTIFACT_DIR / f'confusion_per_label_tuned_{run_id}.csv'
    conf_agg_base_file = ARTIFACT_DIR / f'confusion_aggregate_baseline_{run_id}.csv'
    conf_agg_tuned_file = ARTIFACT_DIR / f'confusion_aggregate_tuned_{run_id}.csv'

    files_exist = all([
        row_file.exists(), per_label_file.exists(), thresholds_file.exists(),
        conf_per_label_base_file.exists(), conf_per_label_tuned_file.exists(),
        conf_agg_base_file.exists(), conf_agg_tuned_file.exists()
    ])

    if SKIP_COMPLETED and files_exist:
        try:
            print(f"[{idx}/{len(LOSS_VARIANTS)}] {run_id}: skipping (already exists)")
            summary_rows.append(pd.read_csv(row_file).iloc[0].to_dict())
            per_label_frames.append(pd.read_csv(per_label_file))
            threshold_frames.append(pd.read_csv(thresholds_file))
            conf_per_label_base_frames.append(pd.read_csv(conf_per_label_base_file))
            conf_per_label_tuned_frames.append(pd.read_csv(conf_per_label_tuned_file))
            conf_agg_base_rows.append(pd.read_csv(conf_agg_base_file).iloc[0].to_dict())
            conf_agg_tuned_rows.append(pd.read_csv(conf_agg_tuned_file).iloc[0].to_dict())
            continue
        except OSError as e:
            print(f"[{idx}/{len(LOSS_VARIANTS)}] {run_id}: Drive read failed ({e}); rerunning this spec.")

    print(f"[{idx}/{len(LOSS_VARIANTS)}] {run_id}: running")

    run_data = preprocess_for_distilbert(
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_val_samples=MAX_VAL_SAMPLES,
        use_iterative_stratify=USE_ITERATIVE_STRATIFY,
        rebalance_train=REBALANCE_TRAIN,
        rebalance_random_state=RANDOM_STATE,
        print_diagnostics=True,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc = enc_dict(run_data.val_encodings)
    y_train = np.asarray(run_data.y_train, dtype=np.float32)
    y_val = np.asarray(run_data.y_val, dtype=np.int64)

    train_loader = DataLoader(
        EncDataset(train_enc, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        EncDataset(val_enc, y_val),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLUMNS),
        problem_type=PROBLEM_TYPE,
    ).to(DEVICE)

    if USE_TORCH_COMPILE and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)  # type: ignore[assignment]
            print('  torch.compile enabled')
        except Exception as e:
            print('  torch.compile skipped:', e)

    num_params = torch_parameter_count(model)

    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

    if spec['use_pos_weight']:
        pos_weight = compute_pos_weight(y_train, max_weight=20.0).to(DEVICE)
        pos_weight_mean = float(pos_weight.detach().cpu().mean().item())
        pos_weight_max = float(pos_weight.detach().cpu().max().item())
    else:
        pos_weight = None
        pos_weight_mean = float('nan')
        pos_weight_max = float('nan')

    def compute_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        if spec['loss_type'] == 'bce':
            return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)
        return focal_bce_with_logits(logits, labels, gamma=float(spec['gamma']), pos_weight=pos_weight)

    steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    num_training_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(WARMUP_RATIO * num_training_steps)
    warmup_steps = max(0, min(warmup_steps, num_training_steps - 1)) if num_training_steps > 0 else 0
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )

    train_time_s = 0.0
    inference_time_s = 0.0
    train_loss_last = float('nan')
    val_loss_last = float('nan')

    best_val_loss = float('inf')
    best_state_cpu = None
    best_epoch = 0
    patience_left = EARLY_STOP_PATIENCE
    epochs_ran = 0
    early_stopped = False

    if USE_AMP:
        autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
    else:
        autocast_ctx = contextlib.nullcontext()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_train_loss = 0.0
        epoch_batches = 0
        t0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        micro = 0

        for batch in train_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                loss = compute_loss(logits, labels) / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            micro += 1
            epoch_train_loss += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
            epoch_batches += 1

            if micro % GRADIENT_ACCUMULATION_STEPS == 0:
                if MAX_GRAD_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        if micro % GRADIENT_ACCUMULATION_STEPS != 0:
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        train_time_s += time.perf_counter() - t0
        train_loss_last = epoch_train_loss / max(epoch_batches, 1)

        model.eval()
        probs_all = []
        val_loss_sum = 0.0
        val_batches = 0
        t1 = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
                labels = batch.pop('labels')
                with autocast_ctx:
                    logits = model(**batch).logits
                    vloss = compute_loss(logits, labels)
                val_loss_sum += float(vloss.item())
                val_batches += 1
                probs_all.append(torch.sigmoid(logits).float().detach().cpu().numpy())

        inference_time_s += time.perf_counter() - t1
        val_loss_last = val_loss_sum / max(val_batches, 1)
        epochs_ran = epoch

        improved = (best_val_loss - val_loss_last) > EARLY_STOP_MIN_DELTA
        if improved:
            best_val_loss = val_loss_last
            best_epoch = epoch
            patience_left = EARLY_STOP_PATIENCE
            best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_left -= 1
            if patience_left <= 0:
                early_stopped = True
                print(f'  early stop at epoch {epoch}')
                break

        print(
            f"  epoch {epoch:02d} | train_loss={train_loss_last:.4f} | val_loss={val_loss_last:.4f} | "
            f"best_val={best_val_loss:.4f}"
        )

    if best_state_cpu is not None:
        model.load_state_dict(best_state_cpu)

    model.eval()
    probs_all = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            _ = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
            probs_all.append(torch.sigmoid(logits).float().detach().cpu().numpy())

    y_prob = np.vstack(probs_all)

    y_pred_base = (y_prob >= 0.5).astype(np.int64)
    _, base_summary = multilabel_evaluation_report(y_val, y_pred_base, y_prob, LABEL_COLUMNS)

    best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, y_prob, LABEL_COLUMNS, THRESHOLD_GRID)
    y_pred_tuned = apply_thresholds(y_prob, LABEL_COLUMNS, best_thresholds)
    per_label_df, tuned_summary = multilabel_evaluation_report(y_val, y_pred_tuned, y_prob, LABEL_COLUMNS)

    per_label_df.insert(0, 'loss_strategy', run_id)

    thresholds_df.insert(0, 'loss_strategy', run_id)

    conf_per_label_base_df, conf_agg_base_df, _ = make_confusion_artifacts(y_val, y_pred_base, LABEL_COLUMNS)
    conf_per_label_base_df.insert(0, 'loss_strategy', run_id)
    conf_agg_base_df.insert(0, 'loss_strategy', run_id)

    conf_per_label_tuned_df, conf_agg_tuned_df, _ = make_confusion_artifacts(y_val, y_pred_tuned, LABEL_COLUMNS)
    conf_per_label_tuned_df.insert(0, 'loss_strategy', run_id)
    conf_agg_tuned_df.insert(0, 'loss_strategy', run_id)

    row = {
        'loss_strategy': run_id,
        'loss_type': spec['loss_type'],
        'focal_gamma': spec['gamma'],
        'use_pos_weight': spec['use_pos_weight'],
        'pos_weight_clip_max': 20.0 if spec['use_pos_weight'] else np.nan,
        'pos_weight_mean': pos_weight_mean,
        'pos_weight_max': pos_weight_max,
        'validation_fraction': VALIDATION_FRACTION,
        'random_state': RANDOM_STATE,
        'use_iterative_stratify': USE_ITERATIVE_STRATIFY,
        'rebalance_train': REBALANCE_TRAIN,
        'model_name': MODEL_NAME,
        'problem_type': PROBLEM_TYPE,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'max_epochs': MAX_EPOCHS,
        'epochs_ran': epochs_ran,
        'best_epoch': best_epoch,
        'early_stopped': early_stopped,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'max_grad_norm': MAX_GRAD_NORM,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'early_stop_min_delta': EARLY_STOP_MIN_DELTA,
        'use_bf16': USE_BF16,
        'use_amp_effective': USE_AMP,
        'use_torch_compile': USE_TORCH_COMPILE,
        'num_workers': NUM_WORKERS,
        'num_params': num_params,
        'train_time_s': train_time_s,
        'inference_time_s': inference_time_s,
        'train_loss_last': train_loss_last,
        'val_loss_last': val_loss_last,
        'baseline_micro_f1': base_summary['f1_micro'],
        'baseline_macro_f1': base_summary['f1_macro'],
        'baseline_samples_f1': float(f1_score(y_val, y_pred_base, average='samples', zero_division=0)),
        'tuned_micro_f1': tuned_summary['f1_micro'],
        'tuned_macro_f1': tuned_summary['f1_macro'],
        'tuned_samples_f1': float(f1_score(y_val, y_pred_tuned, average='samples', zero_division=0)),
        'threshold_grid_start': float(THRESHOLD_GRID[0]),
        'threshold_grid_end': float(THRESHOLD_GRID[-1]),
        'threshold_grid_step': float(THRESHOLD_GRID[1] - THRESHOLD_GRID[0]),
    }

    summary_rows.append(row)
    per_label_frames.append(per_label_df)
    threshold_frames.append(thresholds_df)
    conf_per_label_base_frames.append(conf_per_label_base_df)
    conf_per_label_tuned_frames.append(conf_per_label_tuned_df)
    conf_agg_base_rows.append(conf_agg_base_df.iloc[0].to_dict())
    conf_agg_tuned_rows.append(conf_agg_tuned_df.iloc[0].to_dict())

    pd.DataFrame([row]).to_csv(row_file, index=False)
    per_label_df.to_csv(per_label_file, index=False)
    thresholds_df.to_csv(thresholds_file, index=False)
    conf_per_label_base_df.to_csv(conf_per_label_base_file, index=False)
    conf_per_label_tuned_df.to_csv(conf_per_label_tuned_file, index=False)
    conf_agg_base_df.to_csv(conf_agg_base_file, index=False)
    conf_agg_tuned_df.to_csv(conf_agg_tuned_file, index=False)

    print(
        f"  done {run_id} | tuned_micro_f1={tuned_summary['f1_micro']:.4f} | "
        f"tuned_macro_f1={tuned_summary['f1_macro']:.4f}"
    )

summary_df = pd.DataFrame(summary_rows)
per_label_all_df = pd.concat(per_label_frames, ignore_index=True) if per_label_frames else pd.DataFrame()
thresholds_all_df = pd.concat(threshold_frames, ignore_index=True) if threshold_frames else pd.DataFrame()
conf_per_label_baseline_all_df = pd.concat(conf_per_label_base_frames, ignore_index=True) if conf_per_label_base_frames else pd.DataFrame()
conf_per_label_tuned_all_df = pd.concat(conf_per_label_tuned_frames, ignore_index=True) if conf_per_label_tuned_frames else pd.DataFrame()
conf_agg_baseline_all_df = pd.DataFrame(conf_agg_base_rows)
conf_agg_tuned_all_df = pd.DataFrame(conf_agg_tuned_rows)

summary_df.to_csv(ARTIFACT_DIR / 'sweep_summary.csv', index=False)
per_label_all_df.to_csv(ARTIFACT_DIR / 'sweep_per_label.csv', index=False)
thresholds_all_df.to_csv(ARTIFACT_DIR / 'sweep_thresholds.csv', index=False)
conf_per_label_baseline_all_df.to_csv(ARTIFACT_DIR / 'sweep_confusion_per_label_baseline.csv', index=False)
conf_per_label_tuned_all_df.to_csv(ARTIFACT_DIR / 'sweep_confusion_per_label_tuned.csv', index=False)
conf_agg_baseline_all_df.to_csv(ARTIFACT_DIR / 'sweep_confusion_aggregate_baseline.csv', index=False)
conf_agg_tuned_all_df.to_csv(ARTIFACT_DIR / 'sweep_confusion_aggregate_tuned.csv', index=False)

print('Saved sweep outputs to:', ARTIFACT_DIR)
summary_df.sort_values('tuned_micro_f1', ascending=False).reset_index(drop=True)

In [ ]:
# Quick visualization
if len(summary_df) > 0:
    plot_df = summary_df.sort_values('tuned_micro_f1', ascending=False)
    plt.figure(figsize=(10, 4))
    sns.barplot(data=plot_df, x='loss_strategy', y='tuned_micro_f1', color='steelblue')
    plt.title('Experiment E: tuned micro-F1 by loss strategy')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

summary_df[['loss_strategy', 'loss_type', 'focal_gamma', 'use_pos_weight', 'tuned_micro_f1', 'tuned_macro_f1', 'tuned_samples_f1']].sort_values('tuned_micro_f1', ascending=False).reset_index(drop=True)